[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lbutler2405/EMP5027-rows-to-pixels/blob/main/notebooks/practical-3-structured-data/EMP5027-Lecture-3-Working-with-Structured-Data.ipynb)


# EMP5027: Methods in Data Analysis & Quality Assurance

## Lecture 3: Working with Structured Data

**Instructor:** Dr Liam Butler, University of Malta

Most of the data you will handle as an environmental scientist arrives as a table: a spreadsheet of field measurements, a CSV exported from a logger, a database extract from a monitoring network. This practical is about getting comfortable with that kind of data in pandas, from the first look at a new dataset through to cleaning it, transforming it, and using it in an analysis.

We will work through two datasets that let us focus on the methods rather than on unfamiliar science.

**Datasets used**
- **Penguins** (from `seaborn`): morphological measurements of three penguin species, used throughout Part 1 for cleaning, transformation, and exploratory workflows.
- **Iris** (from `sklearn`): classic plant trait data, used in Part 2 to look at correlation, scaling, and principal component analysis.

Time aware data, sensor time series, resampling, and joining monitoring tables, is covered separately in the companion notebook on time aware data and joins. Here we stay with the two tabular datasets above.

To keep things manageable, we load each dataset once per part and reuse it throughout that part. If you jump into a later section on its own, re run the loading cell for that part first so the variables exist.

## Learning Objectives

By the end of this practical, you should be able to:

- Explain what structured data is and why careful handling of it matters for quality assurance and reproducibility in environmental research.
- Load a tabular dataset and run first pass diagnostics on it: shape, data types, and the number of unique values per column.
- Handle missing data using several strategies (dropping, filling, group based filling, interpolation) and know when each is appropriate, with a brief introduction to model based imputation (MICE).
- Detect and convert data types, find duplicate or invalid values, and flag suspicious rows rather than deleting them outright.
- Engineer new features (ratios, binned categories), group and aggregate data, build pivot tables, sort and rank observations, and detect outliers using the IQR and z-score methods.
- Use string operations, method chaining, and `.pipe()` to write transformations that are easy to read and to audit.
- Examine relationships between features using correlation and covariance, scale features appropriately, and run a principal component analysis (scree plot, PC scatter plot, and loadings).

Time aware data, resampling, and joining tables across sources are covered in the companion notebook on time aware data and joins.

---
# Part 1: Structured Data with Penguins (Ecology Example)

**What is structured data?**

Structured data is any dataset organised into rows and columns, where each row is an observation (a sample, an individual, a site visit) and each column is a variable measured on that observation (temperature, pH, species, location). It is the format behind spreadsheets, CSV files, and relational databases, and it is what makes a dataset easy to query, filter, join, and summarise with a tool like pandas. Because it has this regular shape, structured data also plugs directly into visualisation, statistics, and modelling workflows, which is why so much of quality assurance and quality control in environmental science comes down to handling tables of this kind correctly.

**Environmental relevance**

Field logs (locations, timestamps, weather conditions), sensor time series, laboratory results (E. coli counts, nitrate concentrations), and monitoring reports are all, in practice, structured data. The habits we build in this section, checking shapes and types, deciding deliberately how to handle missing values, flagging rather than silently discarding suspicious readings, apply directly to that kind of real fieldwork data.

## Running this in Google Colab

Click the badge above to open this notebook directly in Colab, no local setup required.
This notebook needs a couple of packages that are not preinstalled on Colab, and the cell below installs them automatically.

`fancyimpute` (used for one optional MICE imputation demo) can be slow or finicky to build, so the notebook works fine without it. That one cell just prints a note and skips the demo instead of failing.

In [ ]:
# --- Google Colab setup (safe to run locally too, it just skips this step) ---
import sys

if "google.colab" in sys.modules:
    !pip install -q fancyimpute

    print("Running in Colab, ready to go.")
else:
    print("Not running in Colab, assuming packages are already installed locally.")

In [ ]:
## Use os package to get the path of your current working directory
import os
os.getcwd()

In [ ]:
# Create a new folder in this path. This will be the folder we will use for this Practical.
os.makedirs("Practical_2a/", exist_ok=True)

In [ ]:
import os
# Confirm the working directory (still the original one, since we haven't changed into the new folder yet)
print("Using os:", os.getcwd())

In [ ]:
# Move into the folder we just created, so any files we save from here land inside it
os.chdir("Practical_2a/")

In [ ]:
import os
# Confirm we're now inside the new folder
print("Using os:", os.getcwd())

In [ ]:
## If os does not work, use pathlib
#from pathlib import Path

# Define the folder you want to create
#new_folder = Path("my_new_folder")

# Create the folder
#new_folder.mkdir(exist_ok=True)

#print(f"Created folder at: {new_folder.resolve()}")

In [ ]:
# --- Load libraries & the Penguins dataset (load ONCE for Part 1) ---
import pandas as pd
import numpy as np

# Seaborn is handy for built-in teaching datasets like 'penguins'
try:
    import seaborn as sns
except ImportError as e:
    raise ImportError("Seaborn is required for the Penguins dataset. Install with `pip install seaborn`.") from e


In [ ]:
df_penguins = sns.load_dataset("penguins")  # seaborn ships this as a built-in teaching dataset, no download needed

# We keep an untouched copy of the raw data. As we clean and transform df_penguins below,
# having df_penguins_raw around means we can always go back and check what changed.
# .copy() matters here: without it, df_penguins_raw would just be another name for the
# same underlying data, and edits to one would silently show up in the other.
df_penguins_raw = df_penguins.copy()

print("Rows, columns:", df_penguins.shape)  # a quick shape check before we look at anything else
df_penguins.head()

In [ ]:
# Write the raw data to disk. index=False avoids adding a spurious "row number" column to the file.
df_penguins_raw.to_csv("Penguins_data.csv", index=False)

In [ ]:
# Read it back in from CSV. This mirrors how you would normally start an analysis, from a file
# on disk rather than from an in-memory object, and lets us check the round trip preserved the data.
df_penguins = pd.read_csv("Penguins_data.csv")
df_penguins

## First Diagnostics

When you receive a new dataset, always start with the same simple checks before doing anything else. They cost almost nothing to run and catch most obvious problems early.
- `.head()` for a quick eyeball preview of the columns and their values.
- `.info()` for each column's dtype and how many non-null values it has, the fastest way to spot missing data and misread types.
- `.describe()` for numeric summary statistics, useful for catching implausible values.
- `.nunique()` to see how many distinct values each column has, which flags categorical variables and helps plan later grouping.

In [ ]:
# .head(10) shows the first 10 rows so we can eyeball column names, units, and obvious problems.
# .info() lists the dtype of each column and how many non-null values it has, which is the fastest
# way to spot missing data and columns that were read in as the wrong type.
display(df_penguins.head(10))
display(df_penguins.info())

In [ ]:
# .describe() gives summary statistics (count, mean, std, min, quartiles, max) for the numeric
# columns, useful for spotting implausible values (e.g. a negative mass) at a glance.
display(df_penguins.describe())

In [ ]:
# .nunique() counts the distinct values in each column. For categorical columns like species,
# island, and sex, this immediately shows how many groups we're working with, which is useful
# for planning group-based analyses later.
display(df_penguins.nunique())  # count unique values per column

## Missing Values: Detect, Decide, Document

Missing values are normal in field and lab data, a sensor drops out, a sample is lost, a form is left blank. What matters is handling them deliberately rather than letting pandas quietly propagate NaN through every downstream calculation.

**Rule of thumb:**
- Drop rows when missing values are few and clearly ignorable.
- Fill values when missingness is systematic or predictable, for example using a group mean.
- Never ignore missing data silently. Document whatever choice you make, since it changes the result.

In [ ]:
# Count missing per column
# .isna() flags each cell as True/False for missing, and .sum() adds those up per column
# (True counts as 1). Sorting descending puts the columns with the most missing data first,
# so we know straight away where to focus.
missing_counts = df_penguins.isna().sum().sort_values(ascending=False)
missing_counts

In [ ]:
# Option A: drop rows with ANY missing values (for clean examples)
# dropna() removes a row if even one of its columns is missing. This is the simplest option,
# but it can throw away a lot of otherwise good data if missingness is spread across columns,
# so we only use it here for a quick clean example, not as a default strategy.
df_penguins_clean = df_penguins.dropna().copy()
print("Original rows:", len(df_penguins))
print("Remaining rows after dropna():", len(df_penguins_clean))
print("Rows dropped:", len(df_penguins) - len(df_penguins_clean))


### Fill Strategies (illustrative)

Below are several common fill strategies applied to different columns, purely to show the syntax and the reasoning behind each one. In a real analysis you would pick one strategy per column based on its context, not apply all of them at once.

In [ ]:
# A1) Mean fill (continuous, roughly normal, no strong group structure)
# Filling with the column mean keeps the dataset complete but slightly reduces the variance of
# the column, since every filled value is exactly the same number. Only reasonable when few
# values are missing and the data don't have obvious subgroup structure.
if 'bill_length_mm' in df_penguins.columns:
    df_penguins_meanfilled = df_penguins.copy()
    df_penguins_meanfilled['bill_length_mm'] = df_penguins_meanfilled['bill_length_mm'].fillna(
        df_penguins_meanfilled['bill_length_mm'].mean()
    )

In [ ]:
df_penguins_meanfilled

In [ ]:
# A2) Median fill (robust to outliers/skew)
# The median is less sensitive to extreme values than the mean, so this is the safer default
# when a column is skewed or has a few very large or very small measurements.
if 'body_mass_g' in df_penguins.columns:
    df_penguins_medianfilled = df_penguins.copy()
    df_penguins_medianfilled['body_mass_g'] = df_penguins_medianfilled['body_mass_g'].fillna(
        df_penguins_medianfilled['body_mass_g'].median()
    )

In [ ]:
df_penguins_medianfilled

In [ ]:
# A3) Group-based fill (preserves biological structure, e.g. species-wise means)
# Instead of filling with one overall mean, we fill each missing value with the mean for that
# individual's own species. This respects the fact that, say, Gentoo penguins are simply heavier
# than Adelie penguins, so a single global mean would be a poor stand-in for either group.
# groupby().transform() is the key idiom here: it returns a result the same length and index as
# the original column, so the filled values slot straight back into df_penguins_groupfilled.
if {'species', 'body_mass_g'}.issubset(df_penguins.columns):
    df_penguins_groupfilled = df_penguins.copy()
    df_penguins_groupfilled['body_mass_g'] = (
        df_penguins_groupfilled
        .groupby('species')['body_mass_g']
        .transform(lambda x: x.fillna(x.mean()))
    )

In [ ]:
df_penguins_groupfilled

In [ ]:
# A4) Interpolation (best for ordered/time series, shown here for syntax)
# interpolate() fills each missing value using a straight line between its neighbouring values.
# That only makes sense when the row order is meaningful, as it is for a time series, so treat
# this as a demonstration of the syntax rather than a sensible choice for this dataset, where
# the row order is arbitrary.
if 'bill_length_mm' in df_penguins.columns:
    df_penguins_interp = df_penguins.copy()
    df_penguins_interp['bill_length_mm'] = df_penguins_interp['bill_length_mm'].interpolate()

In [ ]:
df_penguins_interp

### MICE (Iterative Imputation): A Brief Introduction

The fill strategies above handle one column at a time. MICE (Multiple Imputation by Chained Equations) instead models each column with missing values as a function of the other columns, and iterates until the imputed values stabilise. It is worth knowing about because it makes better use of the correlation between variables (bill length and bill depth, for instance, are related), but it needs an extra package and more computation than a simple fill.

We use `fancyimpute` for this demonstration. It has finicky build dependencies and is not preinstalled on Colab, so the next cell is written to fall back gracefully, printing a note and skipping the imputation, if the package is not available. Read it with diagnostics in mind: model based imputation still produces values that need sense checking against the plausible range of the variable.

In [ ]:
# Optional MICE using fancyimpute (if installed). This is wrapped in a try/except so the cell
# degrades gracefully instead of crashing the whole notebook when fancyimpute isn't available.
# It has finicky build dependencies and isn't preinstalled on Colab, so this guard lets everyone
# run the notebook end to end even without it.
try:
    from fancyimpute import IterativeImputer
    numeric = df_penguins.select_dtypes(include=np.number)
    imputer = IterativeImputer()
    mice_values = imputer.fit_transform(numeric)
    df_penguins_mice = df_penguins.copy()
    df_penguins_mice[numeric.columns] = mice_values
    print("MICE imputation complete (numeric columns only).")
except ImportError:
    print("fancyimpute not installed, skipping MICE imputation. "
          "Install with `pip install fancyimpute` to run this cell.")
    df_penguins_mice = df_penguins.copy()


In [ ]:
df_penguins_mice

## Data Types: Detect and Convert

Real world files often store data with the wrong type: numbers read in as text because of a stray unit or comma, dates stored as plain strings, categorical labels stored as free text. pandas infers a type for each column when it reads a file, but that inference is not always right, and it is worth checking explicitly rather than assuming. Getting the type right matters because it changes what operations are available, you cannot take the mean of a column pandas thinks is text, and how much memory the dataframe uses.

In [ ]:
# List the current dtype of every column, our starting point before converting anything
df_penguins.dtypes

In [ ]:
# Convert a text column to category (saves memory and helps plotting)
# 'sex' only takes a handful of repeated values (Male, Female, and occasionally missing), which
# is exactly what the category dtype is for: pandas stores each unique value once and reuses it,
# rather than repeating the full string for every row. This is more memory efficient and lets
# plotting functions treat it as a proper categorical variable rather than free text.
if 'sex' in df_penguins.columns:
    df_penguins['sex'] = df_penguins['sex'].astype('category')
print(df_penguins['sex'])

In [ ]:
# Force numeric (invalid strings become NaN), useful after messy imports (e.g., Excel)
# errors='coerce' is the important part here: instead of raising an exception the moment it hits
# a value it cannot parse as a number, pandas replaces that value with NaN and carries on. That
# turns a hard failure into a missing value we can inspect and handle deliberately.
df_penguins['body_mass_g'] = pd.to_numeric(df_penguins['body_mass_g'], errors='coerce')
df_penguins.dtypes

## Duplicates and Invalid Values: Flag Before You Drop

Two different problems live under this heading. A duplicate row usually means the same observation was recorded, or read in, more than once. An invalid value is one that is technically present but physically impossible, like a negative body mass or a negative pollutant concentration. In environmental quality assurance the safer habit is to flag suspicious rows in a new column rather than deleting them immediately. That way the decision is visible and reversible, and you, or a reviewer, can always go back and check what was excluded and why.

In [ ]:
# Duplicates
# .duplicated() flags every row that is an exact repeat of an earlier row with True. Summing
# that boolean series counts them, and wrapping in int() gives us a plain number rather than a
# numpy integer type for the print statement.
dup_n = int(df_penguins.duplicated().sum())
print("Duplicated rows:", dup_n)

In [ ]:
# Having confirmed how many duplicates there are, remove them and keep only the first occurrence of each
df_penguins = df_penguins.drop_duplicates()

In [ ]:
df_penguins

In [ ]:
# Example invalid mass: <= 0 grams (physically impossible)
# Build a boolean flag column rather than dropping rows outright. A mass is flagged if it's
# either missing or zero/negative, which combines two separate quality issues into one column
# we can filter on later. The | operator here is elementwise "or" across the two boolean Series.
df_penguins['invalid_mass'] = df_penguins['body_mass_g'].isna() | (df_penguins['body_mass_g'] <= 0)

In [ ]:
df_penguins

In [ ]:
# Demonstration column for a pollutant (not in the original penguins dataset).
# We simulate a nitrate concentration (mg/L) so we have a second, environmentally relevant
# variable to practice invalid-value flagging on, alongside body mass.
rng = np.random.default_rng(0)  # a seeded random generator, so the simulated values are reproducible
df_penguins['nitrate_mg_L'] = rng.normal(loc=0.8, scale=0.3, size=len(df_penguins))
# Deliberately push five random rows negative, since a real nitrate reading can never be negative.
# This gives us guaranteed invalid values to detect below, rather than hoping some appear by chance.
df_penguins.loc[rng.choice(df_penguins.index, size=5, replace=False), 'nitrate_mg_L'] *= -1  # inject negatives
df_penguins['invalid_nitrate'] = df_penguins['nitrate_mg_L'].isna() | (df_penguins['nitrate_mg_L'] < 0)

In [ ]:
print(df_penguins)

In [ ]:
# Confirm the flag only ever takes the two expected boolean values
df_penguins['invalid_nitrate'].unique()

In [ ]:
# Combined flag
# .any(axis=1) checks across the columns (axis=1, i.e. along each row) and returns True if at
# least one of invalid_mass or invalid_nitrate is True for that row. This gives us a single
# column to filter on when we just want "is this row suspect at all", without losing the detail
# of which specific check it failed.
df_penguins['any_invalid'] = df_penguins[['invalid_mass', 'invalid_nitrate']].any(axis=1)
df_penguins['any_invalid']

In [ ]:
df_penguins[['body_mass_g','nitrate_mg_L','invalid_mass','invalid_nitrate','any_invalid']].head(10)

In [ ]:
# value_counts() tallies how many rows fall into each flag category, a quick summary of how
# much of the dataset is affected
print("Invalid Nitrate Readings:", df_penguins['invalid_nitrate'].value_counts())
print("Any Invalid Data:", df_penguins['any_invalid'].value_counts())

## Feature Engineering: Ratios and Bins

Raw measurements are not always the most useful form for plotting or modelling. A ratio between two related measurements can be more informative than either on its own, bill length relative to bill depth says something about shape, not just size, and turning a continuous variable into a small number of bins can make patterns easier to see in a summary table or plot. The goal here is not to replace the original columns, but to add new, interpretable ones alongside them.

In [ ]:
# Ratio: bill length / bill depth
# A single new column that captures bill shape (long and narrow vs short and stout) independent
# of overall bill size, which can be a better predictor of species than either measurement alone.
df_penguins['bill_ratio'] = df_penguins['bill_length_mm'] / df_penguins['bill_depth_mm']
df_penguins

In [ ]:
# Binning body mass into categories (example thresholds)
# pd.cut() slices a continuous variable into discrete intervals defined by 'bins', and assigns
# each row the matching label. The thresholds here (3500 g and 4500 g) are illustrative rather
# than biologically derived cutoffs, but the pattern, pick meaningful breakpoints for your
# variable, is a common way to turn a continuous measurement into a readable category for tables
# and plots.
df_penguins['mass_category'] = pd.cut(
    df_penguins['body_mass_g'],
    bins=[0, 3500, 4500, 6000],
    labels=['Light', 'Medium', 'Heavy']
)

In [ ]:
df_penguins[['bill_length_mm','bill_depth_mm','bill_ratio','body_mass_g','mass_category']].head(10)

## Grouping and Aggregating

`groupby()` is one of the most useful tools in pandas for environmental data: it lets us split the data into subsets defined by a categorical variable (species, sex, site, year) and compute a summary statistic separately for each subset. This is the pandas equivalent of a "summarise by group" step in a lab notebook, and it is how we go from a table of individual observations to the kind of summary numbers that go into a report.

In [ ]:
# Mean mass by species
# groupby('species') splits the rows into one group per species, ['body_mass_g'] selects the
# column we care about within each group, and .mean() computes the average for each group
# separately. The result is a compact summary with one row per species.
mean_mass_by_species = df_penguins.groupby('species')['body_mass_g'].mean()
display(mean_mass_by_species)

In [ ]:
# Multi-aggregation by species & sex
# Grouping by two columns at once (species and sex) gives every combination of the two as a
# group. .agg() then lets us apply several statistics in one call: mean and standard deviation
# for body mass, and the median for flipper length. .reset_index() turns the grouping columns
# back into ordinary columns, which is usually what you want for a table you plan to display or
# export.
agg_df = (
    df_penguins
    .groupby(['species','sex'])
    .agg({'body_mass_g': ['mean','std'], 'flipper_length_mm': 'median'})
    .reset_index()
)
agg_df


## Pivot Tables and Unstacking

A pivot table reshapes long, grouped data into a wide, spreadsheet style summary, with one categorical variable running down the rows and another across the columns. This is often exactly the layout you would build by hand in Excel, and pandas gives you two ways to get there: `.pivot_table()` for building the summary directly, and `.groupby().unstack()` for taking a grouped result you already have and spreading one of its levels out into columns.

In [ ]:
# Minimum body mass by species (rows) and sex (columns)
# pivot_table() takes values to summarise, index for the row grouping, columns for the column
# grouping, and aggfunc for how to summarise each cell. Here aggfunc='min' gives the lightest
# individual recorded for each species/sex combination, rather than the mean.
pivot_mass = df_penguins.pivot_table(
    values='body_mass_g', index='species', columns='sex', aggfunc='min'
)
display(pivot_mass)

In [ ]:
# Grouped maximum flipper length by species & island, unstack island into columns
# groupby(['species','island'])[...].max() first gives a result indexed by both species and
# island together (a MultiIndex). .unstack() then moves the innermost index level, island, out
# into columns, turning that long result into the same kind of wide summary a pivot table gives.
flipper_unstack = (
    df_penguins.groupby(['species','island'])['flipper_length_mm']
    .max()
    .unstack()
)
flipper_unstack

## Sorting and Ranking

Sorting reorders the rows of a dataframe by one or more columns, useful for quickly finding the largest or smallest values, or for presenting a table in a sensible order. Ranking is related but different: instead of reordering the rows, it assigns each one a rank number based on where it falls in the distribution, useful when you want to keep the original row order but still know, for example, which individual was the heaviest.

In [ ]:
# Sort by body mass, lightest first, and preview
# sort_values() reorders the whole dataframe by the given column. ascending=True puts the
# smallest values first, so .head(10) here shows the ten lightest penguins recorded.
sorted_mass = df_penguins.sort_values('body_mass_g', ascending=True).head(10)
display(sorted_mass[['species','island','sex','body_mass_g']])

In [ ]:

# .rank() assigns each row a position in the distribution without reordering the dataframe.
# ascending=False makes the heaviest individual rank 1, and method='average' is how ties are
# broken: rows with an identical mass get the average of the ranks they would otherwise span.
df_penguins['mass_rank'] = df_penguins['body_mass_g'].rank(ascending=False, method='average')
df_penguins[['body_mass_g','mass_rank']]

## Outlier Detection: IQR and Z-score

An outlier is a value that sits unusually far from the rest of the data. That is not automatically a mistake, it could be a measurement error, but it could equally be the most biologically interesting row in the dataset, an unusually large individual, or a genuine pollution spike. The right response is almost never to delete outliers without looking at them. We introduce two common ways to flag them, the interquartile range (IQR) method and the z-score method, so you can compare what each one picks up.

In [ ]:
# IQR method on body_mass_g
# Q1 and Q3 are the 25th and 75th percentiles, the boundaries of the middle 50% of the data.
# The interquartile range (IQR) is the width of that middle chunk, and it's the basis for the
# usual outlier rule: anything more than 1.5 x IQR beyond Q1 or Q3 counts as an outlier.
Q1 = df_penguins['body_mass_g'].quantile(0.25)
Q3 = df_penguins['body_mass_g'].quantile(0.75)
IQR = Q3 - Q1
IQR

In [ ]:
# Apply the standard 1.5 x IQR rule: flag rows that fall either well below Q1 or well above Q3.
# This threshold is a convention, not a law of nature, but it is a widely used starting point.
outliers_iqr = df_penguins[(df_penguins['body_mass_g'] < Q1 - 1.5*IQR) | (df_penguins['body_mass_g'] > Q3 + 1.5*IQR)]
print("IQR outliers:", len(outliers_iqr))
outliers_iqr.head(5)

In [ ]:
# Z-score method
# A z-score expresses each value as the number of standard deviations it sits from the mean,
# so it is directly comparable across variables with different units or scales.
from scipy.stats import zscore
df_penguins['z_mass'] = zscore(df_penguins['body_mass_g'].astype(float))
# Fallback if SciPy is not available: compute the same thing by hand. ddof=0 matches SciPy's
# default of dividing by n rather than n-1, so the two versions agree.
col = df_penguins['body_mass_g'].astype(float)
df_penguins['z_mass'] = (col - col.mean()) / col.std(ddof=0)

In [ ]:
# A common convention flags anything more than 2 standard deviations from the mean. It's a
# looser threshold than 3 SD, so expect it to flag somewhat more rows than a stricter cutoff would.
outliers_z = df_penguins[df_penguins['z_mass'].abs() > 2]
print("Z-score outliers (>2 SD):", len(outliers_z))
outliers_z

## `.apply()` vs `np.where()`

Both of these let you create a new column based on a condition, but they suit different situations. `.apply()` runs a Python function on every value, so it can express any logic you like, including multiple conditions with if/elif/else, at the cost of being slower on large datasets because it is not vectorised. `np.where()` is a fast, vectorised way to answer a single yes or no question across the whole column at once. Use `.apply()` when the logic genuinely needs branching, and `np.where()` when it is a simple binary split.

In [ ]:
# Classify flipper length into three bands using an ordinary Python function. This is exactly
# the kind of multi-branch logic .apply() is suited to: three possible outcomes based on where
# the value falls, which np.where() alone cannot express as cleanly.
def classify_flipper(length: float) -> str:
    if length > 210: 
        return "Long"
    elif length < 185:
        return "Short"
    else:
        return "Medium"

In [ ]:
# .apply() calls classify_flipper() once per row, passing in that row's flipper_length_mm value
df_penguins['flipper_class'] = df_penguins['flipper_length_mm'].apply(classify_flipper)
df_penguins['flipper_class']

In [ ]:
# Binary 'heavy' using np.where
# np.where(condition, value_if_true, value_if_false) is vectorised: it evaluates the condition
# across the whole column at once rather than row by row, which makes it much faster than
# .apply() for a simple binary flag like this.
df_penguins['is_heavy'] = np.where(df_penguins['body_mass_g'] > 4500, 1, 0)
df_penguins['is_heavy']

In [ ]:
df_penguins[['flipper_length_mm','flipper_class','body_mass_g','is_heavy']].head(10)

## String Operations

Text columns in real datasets are rarely as clean as they look. Inconsistent capitalisation, stray whitespace, and typos are common, especially in anything that was typed by hand in the field or copied between spreadsheets. pandas exposes the usual string methods (lower case, strip, contains, replace) through the `.str` accessor, which applies them across an entire column at once rather than one value at a time.

In [ ]:
# Work on a copy so we can compare the cleaned text against the original column below,
# rather than overwriting df_penguins.
df_text = df_penguins.copy()
# .str.lower() lowercases every value in the column in one call, useful for making text
# comparisons and filters case-insensitive.
df_text['island'] = df_text['island'].str.lower()

In [ ]:
# Compare the original capitalisation against the lower-cased version
print(df_penguins['island'])
print(df_text['island'])

In [ ]:
# .str.strip() removes leading and trailing whitespace, which is invisible in a printed table
# but can silently break exact-match filtering and grouping if it's present.
df_text['species'] = df_text['species'].str.strip()
print(df_text['species'])

In [ ]:
# Examples
# .str.contains() filters rows where the column contains a given substring. na=False treats
# any missing values as "no match" rather than raising an error, which is the safer default.
contains_dream = df_text[df_text['island'].str.contains('dream', na=False)]
# .str.startswith() checks the beginning of each string. We lower-cased 'island' above but not
# 'species', so this comparison against the capitalised 'Ade' is still checking the original case.
starts_ade = df_text[df_text['species'].str.startswith('Ade', na=False)]  # none expected, illustrative
# .str.replace() does a straightforward find-and-replace. regex=False tells pandas to treat
# 'biscoe' as a literal substring rather than a regular expression pattern, and case=False makes
# the match case-insensitive even though we already lower-cased this column.
fixed_island = df_text['island'].str.replace('biscoe', 'Biscoe Island', case=False, regex=False)

In [ ]:
## Print out the three objects you created and see what the differences are from the original



In [ ]:
## Print out the three objects you created and see what the differences are from the original



In [ ]:
## Print out the three objects you created and see what the differences are from the original



In [ ]:
# Preview the rows where the island name contains 'dream'
display(contains_dream.head(3))

In [ ]:
# Preview the replaced island column
display(fixed_island)

## Clean Pipelines with Method Chaining and `.pipe()`

Once you have several transformation steps to apply in sequence, it becomes easy to lose track of what happened in what order if each step is a separate line reassigning the same variable. Method chaining, stringing operations together with dots, and `.pipe()`, which lets you insert a custom function into that chain, keep the whole sequence readable top to bottom and easy to audit later. This matters for reproducibility: a chained pipeline reads almost like a list of instructions, which makes it much easier for someone else, or you in six months, to see exactly what was done to the data.

In [ ]:
# A small function written so it can slot into a .pipe() call: it takes a dataframe in and
# returns a dataframe out. Working on a copy (out = df.copy()) means calling this function never
# modifies the dataframe that was passed in, which is what keeps a chained pipeline predictable.
def add_ratio(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['bill_ratio'] = out['bill_length_mm'] / out['bill_depth_mm']
    return out

In [ ]:
# Read this chain top to bottom as a sequence of steps: start from df_penguins, drop any rows
# with missing values, pass the result through add_ratio() via .pipe(), then group by species and
# island and average the new bill_ratio column. Nothing here reassigns an intermediate variable,
# so there's no risk of accidentally reusing a stale version of the data partway through.
ratio_by_species = (
    df_penguins.dropna()
    .pipe(add_ratio)
    .groupby(['species','island'])['bill_ratio']
    .mean()
)
ratio_by_species


## Visualisation (Exploratory Data Analysis)

Numbers alone can hide patterns that are obvious in a plot: a distribution's skew, a cluster of points, a handful of outliers sitting away from the rest. A few quick plots here are not a substitute for the checks we have already done, but they are an easy way to confirm that what the summary statistics suggested is actually visible in the shape of the data.

In [ ]:
# Histogram + KDE (requires seaborn)
# A histogram bins body mass and shows how many observations fall in each bin, while the KDE
# (kernel density estimate) overlay is a smoothed version of the same distribution. Together
# they make it easy to see whether the data are roughly symmetric or skewed, and whether there
# are one or several peaks. We drop missing values first since histplot cannot plot a NaN.
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure()
sns.histplot(df_penguins['body_mass_g'].dropna(), kde=True)
plt.title("Body Mass Distribution")
plt.xlabel("Body Mass (g)")
plt.show()

In [ ]:
# A boxplot per species shows the median, interquartile range, and any points flagged as
# outliers (beyond the whiskers) for each group side by side, a quick visual check against the
# IQR outlier flags we computed numerically earlier.
plt.figure()
sns.boxplot(data=df_penguins, x='species', y='body_mass_g')
plt.title("Mass by Species")
plt.show()


In [ ]:
# Colouring by species (hue='species') often reveals that a relationship which looks weak or
# absent in the data as a whole is actually clear and consistent within each species, a reminder
# to check for grouping structure before concluding two variables are unrelated.
plt.figure()
sns.scatterplot(data=df_penguins, x='bill_length_mm', y='bill_depth_mm', hue='species')
plt.title("Bill Length vs Depth by Species")
plt.xlabel("Bill Length/mm")
plt.ylabel("Bill Depth/mm")
plt.show()

In [ ]:
## Save the cleaned and transformed Penguins dataset
# Writing out the working dataframe, with all the flags and engineered columns we've added,
# keeps the cleaning decisions we made in this notebook, rather than leaving them implicit.
df_penguins.to_csv("Updated_Penguins_Data.csv", index=False)

---
# Part 2: Correlation and Feature Analysis with Iris

We now switch to the Iris dataset, four measurements taken on three species of iris flower, to look at how features relate to one another. This is a smaller, entirely numeric dataset, which makes it a clean example for correlation, covariance, feature scaling, and principal component analysis (PCA), techniques you will use often once you move from cleaning environmental data to actually analysing it.

In [ ]:
# sklearn ships Iris as a Bunch object rather than a dataframe: iris_raw.data holds the raw
# numeric measurements as a numpy array, and iris_raw.feature_names holds the matching column
# names. Printing the first five rows and the feature names lets us see what we're working with
# before we build a proper dataframe from them.
from sklearn.datasets import load_iris

iris_raw = load_iris()
print(iris_raw.data[0:5])
print(iris_raw.feature_names)

In [ ]:
# Wrap the numpy array in a DataFrame with the feature names as column headers, which gives us
# all the familiar pandas methods to work with from here on
df_iris = pd.DataFrame(iris_raw.data, columns=iris_raw.feature_names)
df_iris

In [ ]:
# target holds each row's species as an integer code (0, 1, 2), and target_names maps those
# codes to the actual species names
iris_raw.target, iris_raw.target_names

In [ ]:
# pd.Categorical.from_codes() turns the integer codes back into readable species labels in one
# step, and adding them as a category column keeps the memory footprint small compared to
# storing the species name as a plain string on every row
df_iris['species'] = pd.Categorical.from_codes(iris_raw.target, iris_raw.target_names)
df_iris

## Correlation Matrix

Correlation measures the strength and direction of a linear relationship between two variables, on a scale from -1 (a perfect negative relationship) through 0 (no linear relationship) to +1 (a perfect positive relationship). It says nothing about causation, and it can miss a strong relationship that is not a straight line, but it is a fast first check for which pairs of features move together, which matters before running something like PCA that is sensitive to correlated inputs.

In [ ]:
# .iloc[:, :-1] takes every column except the last, which drops the categorical 'species'
# column so .corr() only sees the four numeric measurements. By default .corr() computes the
# Pearson correlation coefficient between every pair of columns.
cor_matrix = df_iris.iloc[:, :-1].corr()
cor_matrix

In [ ]:
# Heatmap
# A heatmap makes a correlation matrix much easier to scan than a table of numbers: colour
# intensity shows the strength of each relationship at a glance. annot=True prints the actual
# coefficient in each cell too, and fmt=".2f" rounds it to two decimal places.
plt.figure()
sns.heatmap(cor_matrix, annot=True, fmt=".2f")
plt.title("Correlation Matrix (Iris)")
plt.show()

## Pairwise Relationships (`pairplot`)

A correlation matrix tells you the strength of a linear relationship, but a pairplot shows you the relationship itself: every feature plotted against every other feature, with the diagonal showing each feature's own distribution. Colouring by species lets us see whether the species form distinct clusters in this feature space, which the correlation numbers alone cannot show, and it is often the fastest way to spot a non linear pattern that a correlation coefficient would miss entirely.

In [ ]:
# sns.pairplot() builds the full grid of scatterplots automatically from every numeric column
# in df_iris. hue='species' colours each point by species, and diag_kind='hist' shows a histogram
# rather than a density curve on the diagonal, where a feature is plotted against itself.
plt.figure()
sns.pairplot(df_iris, hue='species', diag_kind='hist')
plt.show()

## Covariance Matrix

Covariance measures the same kind of joint variability as correlation, but it is not standardised, so its value depends on the units and scale of the underlying variables and cannot be compared directly across different pairs of features. It matters here mainly as groundwork for PCA, which is built directly on the covariance, or correlation, structure of the data.

In [ ]:
# Same idea as the correlation matrix above, but .cov() returns the raw covariance rather than
# the standardised correlation coefficient, so the magnitudes here reflect the original
# measurement units (cm) and are not directly comparable across feature pairs
cov_matrix = df_iris.iloc[:, :-1].cov()
cov_matrix


In [ ]:
# Same heatmap approach as before, though note the values here (and the colour scale) aren't
# on the same -1 to +1 scale as the correlation heatmap, since covariance isn't standardised
plt.figure()
sns.heatmap(cov_matrix, annot=True, fmt=".1f")
plt.title("Covariance Matrix (Iris)")
plt.show()

## Feature Scaling

The four Iris measurements are all in centimetres, so they happen to be on a similar scale already, but that will not be true in general. Environmental datasets routinely mix variables measured in wildly different units and ranges, a temperature in degrees, a concentration in parts per million, a count in the thousands, and any method that relies on distance or variance, PCA, clustering, many machine learning models, will be dominated by whichever variable happens to have the largest raw numbers unless we scale first.

Two common approaches:
- **Min-Max (0 to 1)** compresses each feature into a bounded range.
- **Standardisation (mean 0, std 1)** rescales each feature relative to its own spread.

PCA specifically calls for standardisation, since it is built on variance and we do not want that variance driven purely by units.

In [ ]:
df_iris

In [ ]:
# MinMaxScaler rescales each column independently so its minimum value becomes 0 and its
# maximum becomes 1. fit_transform() learns the min and max from this data and applies the
# rescaling in one step. We again drop the 'species' column with .iloc[:, :-1] since scaling only
# makes sense for the numeric features.
from sklearn.preprocessing import MinMaxScaler, StandardScaler

minmax = MinMaxScaler().fit_transform(df_iris.iloc[:, :-1])
pd.DataFrame(minmax)

In [ ]:
# StandardScaler instead rescales each column to have mean 0 and standard deviation 1. This is
# the version we'll carry forward into the PCA below.
standardised = StandardScaler().fit_transform(df_iris.iloc[:, :-1])
pd.DataFrame(standardised)

In [ ]:
# Reattach the original column names, lost when the scalers returned plain numpy arrays, so the
# scaled values are still labelled
pd.DataFrame(minmax, columns=df_iris.columns[:-1]).head()

In [ ]:
# Same relabelling for the standardised version
pd.DataFrame(standardised, columns=df_iris.columns[:-1]).head()


## Principal Component Analysis (PCA)

PCA takes a set of correlated features and re-expresses them as a new set of uncorrelated variables, the principal components, ordered so that the first component captures as much of the variance in the data as possible, the second captures as much of what is left, and so on. It is a workhorse technique in environmental science for reducing a large number of correlated measurements (soil chemistry variables, for instance, or spectral bands) down to a handful of components that are easier to plot, interpret, and use in further analysis. We will look at three things in turn: how much variance each component explains, how the samples look when plotted in the space of the first two components, and which original features contribute most to each component.

In [ ]:
# PCA() with no arguments keeps every component (as many as there are input features). Fitting
# it on the standardised data, not the raw measurements, is what makes the components reflect
# genuine patterns in the data rather than an artefact of the original units.
from sklearn.decomposition import PCA
import numpy as np

pca = PCA()
# fit_transform() both learns the components and projects every sample onto them in one step,
# giving us the coordinates of each sample in the new, uncorrelated component space.
X_pca = pca.fit_transform(standardised)
X_pca_df = pd.DataFrame(X_pca)
X_pca_df

In [ ]:
# Scree plot (cumulative explained variance)
# explained_variance_ratio_ gives the fraction of total variance each component accounts for,
# in descending order. np.cumsum() adds those up as a running total, so this plot shows how much
# of the original variance we would retain if we kept only the first N components, useful for
# deciding how many components are actually worth carrying forward.
plt.figure()
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o')
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Scree Plot: PCA Explained Variance")
plt.grid(True)
plt.show()

In [ ]:
# First two PCs scatter coloured by species
# X_pca[:, :2] takes only the first two columns of the PCA output, the first two principal
# components, since those two alone usually capture most of the variance and are what we can
# actually plot on a 2D scatterplot. We reattach the species labels afterwards purely for
# colouring the plot below, they play no part in how PCA itself was computed.
df_pca = pd.DataFrame(X_pca[:, :2], columns=['PC1','PC2'])
df_pca['species'] = df_iris['species']
df_pca

In [ ]:
# If the species separate cleanly into distinct clusters here, that's a sign the original four
# measurements do capture real, consistent differences between them, in just two components
plt.figure()
sns.scatterplot(data=df_pca, x='PC1', y='PC2', hue='species')
plt.title("Iris in PCA Space (PC1 vs PC2)")
plt.show()


In [ ]:
# PCA loadings (feature contributions to each PC)
# pca.components_ holds, for each principal component, the weight given to every original
# feature when building that component. A large positive or negative loading means that feature
# strongly influences the component, while a value near zero means it barely contributes. This
# is how we interpret what a component actually represents in terms of the original measurements,
# rather than treating it as an abstract number.
loadings = pd.DataFrame(
    pca.components_,
    columns=df_iris.columns[:-1],
    index=[f'PC{i+1}' for i in range(len(pca.components_))]
)
loadings.head()


---
## Practice (Suggested)

1. Recreate the IQR and z-score outlier flags for `flipper_length_mm` and compare which rows each method flags.
2. Build a method-chained pipeline that drops missing values, adds `bill_ratio`, groups by species and sex, and computes the mean `bill_ratio` for each group.
3. Repeat the PCA workflow on a subset of the Iris features (drop one measurement) and compare the resulting scree plot and PC scatter plot to the full four-feature version. What changes, and what does that tell you about which feature was contributing least?

### Notes

- Always log your imputation and flagging decisions. A cleaned dataset without a record of what was changed and why is not reproducible.
- Prefer transparent preprocessing, code with comments, over hidden spreadsheet steps that nobody can retrace.
- Keep the raw data read-only and build your cleaned version as a separate object or file, the way we kept `df_penguins_raw` alongside `df_penguins` throughout this notebook.